# CacaoScan - Entrenamiento de modelos en Google Colab

Pipeline:
1. Clona el repo
2. Instala dependencias (runtime + training)
3. Descarga dataset desde S3 (o sube manualmente a `/content/data`)
4. Entrena U-Net background → calibra pixeles → entrena regresor hibrido
5. Sube los `.pth/.pt` resultantes al bucket S3 de modelos

**Antes de correr:** Runtime → Cambiar tipo de entorno → GPU (T4 o L4).

## 1. Configuracion (edita estas variables)

In [ ]:
REPO_URL = 'https://github.com/JefersonCCJM/cacaoscan.git'
BRANCH = 'develop'

# Buckets S3 (o MinIO si tienes tunel)
S3_DATASETS_BUCKET = 'cacaoscan-datasets'
S3_MODELS_BUCKET = 'cacaoscan-models'
AWS_REGION = 'us-east-1'

# Version del modelo a generar (SemVer en el path del bucket)
MODEL_VERSION = 'v0.1.0'

# Hiperparametros
UNET_EPOCHS = 20
UNET_BATCH = 16
HYBRID_EPOCHS = 50
HYBRID_BATCH = 32

In [ ]:
# Credenciales AWS - usa Colab Secrets (icono de llave en barra izquierda)
# Crea 2 secrets: AWS_ACCESS_KEY_ID y AWS_SECRET_ACCESS_KEY
from google.colab import userdata
import os
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION

## 2. Clonar repo + instalar dependencias

In [ ]:
!nvidia-smi
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/cacaoscan
%cd /content/cacaoscan/backend

In [ ]:
# Torch ya viene en Colab con CUDA. Instalamos el resto (saltando torch).
!grep -v -E '^(torch|torchvision)' requirements.txt > /tmp/reqs.txt
!pip install -q -r /tmp/reqs.txt -r requirements-train.txt awscli

## 3. Descargar dataset desde S3

In [ ]:
!mkdir -p media/cacao_images/raw media/datasets
!aws s3 sync s3://{S3_DATASETS_BUCKET}/raw/cacao_images/ media/cacao_images/raw/
!aws s3 sync s3://{S3_DATASETS_BUCKET}/processed/ media/datasets/
!ls -lah media/cacao_images/raw | head

## 4. Configurar Django (DB SQLite efimera para training)

In [ ]:
import os
os.environ['DJANGO_SETTINGS_MODULE'] = 'cacaoscan.settings'
os.environ['APP_ENV'] = 'training'
os.environ['SECRET_KEY'] = 'colab-training-not-for-prod'
os.environ['DEBUG'] = 'False'
os.environ['USE_REDIS'] = 'False'
os.environ['USE_CELERY_REDIS'] = 'False'
os.environ['USE_S3'] = 'False'
# DB SQLite efimera (training no necesita Postgres)
os.environ['DB_ENGINE'] = 'sqlite'

!python manage.py migrate --run-syncdb 2>&1 | tail -5

## 5. Entrenar U-Net (segmentacion de fondo)

In [ ]:
!python manage.py train_unet_background --epochs {UNET_EPOCHS} --batch-size {UNET_BATCH}

## 6. Calibrar pixeles

In [ ]:
!python manage.py calibrate_dataset_pixels --segmentation-backend auto

## 7. Entrenar regresor hibrido

In [ ]:
!python manage.py train_cacao_models --hybrid --use-pixel-features --epochs {HYBRID_EPOCHS} --batch-size {HYBRID_BATCH}

## 8. Subir artefactos versionados a S3

In [ ]:
import json, datetime

manifest = {
    'version': MODEL_VERSION,
    'trained_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'branch': BRANCH,
    'hyperparams': {
        'unet_epochs': UNET_EPOCHS,
        'unet_batch': UNET_BATCH,
        'hybrid_epochs': HYBRID_EPOCHS,
        'hybrid_batch': HYBRID_BATCH,
    },
}
with open('manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

DEST = f's3://{S3_MODELS_BUCKET}/{MODEL_VERSION}'
!aws s3 cp ml/segmentation/cacao_unet.pth {DEST}/segmentation/cacao_unet.pth
!aws s3 cp ml/artifacts/regressors/hybrid.pt {DEST}/regression/hybrid.pt
!aws s3 cp media/datasets/pixel_calibration.json {DEST}/calibration/pixel_calibration.json
!aws s3 cp manifest.json {DEST}/manifest.json
print(f'\nModelos subidos a {DEST}/')